## Overview: data preparation for commits regressions 

Note: the resulting tables (commits, users, projects) are rather big. Therefore, all the preliminary preparation, cleaning and merging is done on the Cloud. Only clean and short task-specific tables are used in Stata to run regressions and to prepare summary statistics. 

* Regression preparations 
    * Baseline -- commits aggregated at a country pair level, excluding users from DLK


In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

In [ ]:
# example codes 
'''

# check processing costs
job = client.query(query, job_config=dry)
print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
print("Bytes billed: ", job.total_bytes_billed)

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.head()

In [ ]:
#extraction for the baseline survey
#projects temp table (use info from first parents if forked)
#projects_commits temp table (with owner info)
#puc temp table (commits with owner and author info) 
# --> group by country_owner, country_author, year, month (baseline)
# --> if forked = 0: robustness if only direct contributions
# --> if nmloc (stayers) = 1: robustness to stayers

In [ ]:
#projects: project_id, dummy if forked, owner_id (linked on project_id)
query='''
CREATE VIEW `friends-foes.commits.projects_tomerge_noforks`
AS
SELECT project_id, forked_from, forked, parent_id, owner_id
FROM
    (SELECT id AS project_id, forked_from,  
        CASE
         WHEN forked_from IS NULL THEN 0
         ELSE 1
        END forked, 
        CASE
         WHEN forked_from IS NULL THEN id
         ELSE forked_from
        END parent_id     
    FROM `friends-foes.commits.projects`) pids
LEFT JOIN 
# join owner/project info on project id
    (SELECT id, owner_id, 
    EXTRACT(YEAR FROM created_at) as project_year, EXTRACT(MONTH FROM created_at) as project_month,
    FROM `friends-foes.commits.projects`) oids
ON oids.id = pids.project_id
'''
client.query(query, job_config=run) 

In [ ]:
# merge commits with projects to find owner location
query='''
CREATE VIEW `friends-foes.commits.commits_projects_noforks`
AS 
SELECT c.year, c.month, c.author_id, c.project_id, commits, 
       forked, parent_id, owner_id, country_current AS country_owner, dlk AS dlk_owner,
       ru_identity AS ru_identity_owner, district_en AS district_owner, name_class AS name_class_owner
FROM 
(SELECT * FROM `friends-foes.commits.commits`) c
LEFT JOIN `friends-foes.commits.projects_tomerge_noforks` p
ON c.project_id = p.project_id
LEFT JOIN `friends-foes.user_panel.sample_locations_panel` ul
ON p.owner_id = ul.id AND c.year = ul.year 
LEFT JOIN `friends-foes.user_panel.ua_users` ua 
ON p.owner_id = ua.id AND c.year = ua.year 
LEFT JOIN `friends-foes.user_panel.name_class` cl 
ON p.owner_id = cl.id
#WHERE c.author_id <> p.owner_id #comment out if want to drop self-commits
'''
client.query(query, job_config=run) 

In [ ]:
# merge with users to find author location
query='''
DROP TABLE IF EXISTS `friends-foes.commits.puc_noforks`; 

CREATE TABLE `friends-foes.commits.puc_noforks`
AS
SELECT c.year AS year, month, author_id, project_id, commits, 
       forked, parent_id, owner_id, country_owner, 
       dlk_owner, ru_identity_owner, district_owner, name_class_owner,
       country_current AS country_author, dlk AS dlk_author, 
       ru_identity AS ru_identity_author, district_en AS district_author, 
       name_class as name_class_author
FROM `friends-foes.commits.commits_projects_noforks` c
LEFT JOIN `friends-foes.user_panel.sample_locations_panel` ul
ON c.author_id = ul.id AND c.year = ul.year 
LEFT JOIN `friends-foes.user_panel.ua_users` ua 
ON c.author_id = ua.id AND c.year = ua.year 
LEFT JOIN `friends-foes.user_panel.name_class` cl 
ON c.author_id = cl.id
'''
client.query(query, job_config=run) 

In [ ]:
# baseline regression
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT project_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc_noforks`
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author
'''
job = client.query(query, job_config=dry)
print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT project_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc_noforks`
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_nofork_wself.csv', encoding = 'utf-8', index=False)

In [ ]:
query='''
SELECT year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc_noforks`
GROUP BY year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_ident.csv', encoding = 'utf-8', index=False)

In [ ]:
# robustness for users who joined before 2014
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects,     COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc_noforks` 
WHERE author_id IN (SELECT DISTINCT id FROM `friends-foes.user_panel.sample_locations_panel`
                    WHERE year_created < 2014) 
AND 
owner_id IN (SELECT DISTINCT id  FROM `friends-foes.user_panel.sample_locations_panel`
                  WHERE year_created < 2014) 
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013-s.csv', encoding = 'utf-8', index=False)

In [ ]:
# robustness for users who joined before 2014 and with reported locations as of 2013
query='''
SELECT year, month, cc2013_author, cc2013_owner, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners, MAX(year_created_o), MAX(year_created_a)
FROM `friends-foes.commits.puc_noforks` c
INNER JOIN (SELECT DISTINCT id, year_created AS year_created_a, 
            CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #for those with missing country codes 
            END cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            WHERE year_created < 2014 AND year = 2013) up
ON c.author_id = up.id
INNER JOIN (SELECT DISTINCT id, year_created AS year_created_o, 
            CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then country_current #for those with missing country codes
            END cc2013_owner
            FROM `friends-foes.user_panel.sample_locations_panel`
            WHERE year_created < 2014 AND year = 2013) op
ON c.owner_id = op.id
GROUP BY year, month, cc2013_owner, cc2013_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013_noforks.csv', encoding = 'utf-8', index=False)

In [ ]:
# movers vs stayers 
query='''
CREATE VIEW `friends-foes.commits.movers`
AS
SELECT DISTINCT author_id, 
#don't count if moved in 2019 as not part of the sample 
CASE WHEN cc2013 = cc2014 AND cc2013 = cc2016 AND cc2013 = cc2019 THEN 1 ELSE 0 END stayer,
CASE WHEN  n_countries > 1 THEN 1 ELSE 0 END mover,
#CASE WHEN cc2013 = 'ru' AND (cc2014 = 'ua' OR cc2016 = 'ua' OR cc2019 = 'ua') THEN 1 ELSE 0 END mover_ru_ua,
#CASE WHEN cc2013 = 'ua' AND (cc2014 = 'ru' OR cc2016 = 'ru' OR cc2019 = 'ru') THEN 1 ELSE 0 END mover_ua_ru
FROM (SELECT DISTINCT id as author_id, cc2013, cc2014, cc2016, cc2019, 
      COUNT(DISTINCT country_current) OVER(PARTITION BY id) as n_countries 
      FROM `friends-foes.user_panel.sample_locations_panel`
      WHERE year < 2020
      ) #only until 2019
'''
job = client.query(query, job_config=run) 

In [ ]:
# movers vs stayers (stayers for both owner and author)
query='''
SELECT year, month, cc2013_author, cc2013_owner, dlk_owner, dlk_author,  
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT c.author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners, 
       stayer_owner, stayer_author,
       MAX(year_created_o), MAX(year_created_a)
FROM `friends-foes.commits.puc_noforks` c
LEFT JOIN (SELECT author_id, stayer AS stayer_author FROM `friends-foes.commits.movers`) ma
ON c.author_id = ma.author_id
LEFT JOIN (SELECT author_id, stayer AS stayer_owner FROM `friends-foes.commits.movers`) mo
ON c.owner_id = mo.author_id
INNER JOIN (SELECT DISTINCT id, year_created AS year_created_a, 
            CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then cc2016 #for control countries only
            END cc2013_author
            FROM `friends-foes.user_panel.sample_locations_panel`
            WHERE year_created < 2014) up
ON c.author_id = up.id
INNER JOIN (SELECT DISTINCT id, year_created AS year_created_o, 
            CASE
                WHEN cc2013 IN ('ua', 'ru') THEN cc2013
                WHEN cc2013 is null then cc2016 #for control countries only
            END cc2013_owner
            FROM `friends-foes.user_panel.sample_locations_panel`
            WHERE year_created < 2014) op
ON c.owner_id = op.id
GROUP BY year, month, cc2013_owner, cc2013_author, dlk_owner, dlk_author, stayer_owner, stayer_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013-stayers.csv', encoding = 'utf-8', index=False)

In [ ]:
# repeat with push-events from GH torrent + in real time
query = 
'''
  SELECT year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author,
           name_class_owner, name_class_author,
           SUM(events) AS events_int, COUNT(events) AS events_ext, COUNT(DISTINCT author_login) AS n_authors, 
           COUNT(DISTINCT repo_url) AS n_projects
  FROM
  (SELECT *
  FROM 
    (SELECT EXTRACT(YEAR FROM created_at) AS year, EXTRACT(MONTH FROM created_at) AS month, actor.login as author_login, repo.url,
       CASE WHEN STRPOS(repo.url, 'https://api.github.com/repos/') > 0
                THEN SUBSTR(repo.url, LENGTH('https://api.github.com/repos/')+1, LENGTH(repo.url)) 
            WHEN STRPOS(repo.url, 'https://github.com/') > 0
                THEN SUBSTR(repo.url, LENGTH('https://github.com/')+1, LENGTH(repo.url)) 
            ELSE "no repo info" 
            END AS repo_url, 
       SUM(*) AS events
    FROM `githubarchive.month.2020*`
    WHERE  type='PushEvent' 
    GROUP BY year, month, actor_login, repo.url, repo_url
    )
  WHERE STRPOS(repo_url, '/') > 0 AND author_login NOT LIKE SUBSTR(repo_url, 0, (STRPOS(repo_url, '/')-1)) #owner and committer are different
  )
INNER JOIN 
    (SELECT login, country_code AS country_c, city AS city_c, long AS long_c, lat AS lat_c, 
    CASE WHEN company IS NULL THEN 0
    ELSE 1 
    END AS company_c 
    FROM `identity-programmers.ghtorrent2020.gh_usersJun2020` WHERE country_code != '\\N') c
ON actor_login = c.login
INNER JOIN (SELECT login, country_code AS country_o, city AS city_o, long AS long_o, lat AS lat_o, 
CASE WHEN company IS NULL THEN 0
ELSE 1 
END AS company_o 
FROM `identity-programmers.ghtorrent2020.gh_usersJun2020`  WHERE country_code != '\\N') o
ON SUBSTR(repo_url, 0, (STRPOS(repo_url, '/')-1)) = o.login
GROUP BY year, month, long_c, lat_c, long_o, lat_o, city_c, city_o, country_c, country_o
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_gharchive.csv', encoding = 'utf-8', index=False)
client.query(query, job_config=run) 
